# 1+1 CDT spacetime visualization

This notebook generates an example visualization for a small 1+1-dimensional CDT run. It uses the Rust `cdt` binary as the simulation engine, reads the generated trace and summary artifacts, then renders the final exported spacetime triangulation.

Summaries pass through Delaunay's stable `final_triangulation.mesh` interchange value: vertices and simplices carry UUIDs, and simplex connectivity uses vertex UUIDs rather than compact indices. CDT foliation labels are joined from `final_triangulation.vertex_time_labels` by UUID. Profile-only summaries cannot reconstruct connectivity and must be regenerated with the current schema.

## 1. Setup

From the repository root, run `just notebook-setup` once before opening this notebook. That installs the uv-managed notebook dependency group. The `justfile` contains the exact commands if you want to inspect what it does.

In [ ]:
import json
import math
import os
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any
from uuid import UUID

import matplotlib.pyplot as plt
import polars as pl
from matplotlib.collections import LineCollection, PolyCollection

UTF8 = "utf-8"


@dataclass(frozen=True)
class RunPaths:
    output_dir: Path
    trace_csv: Path
    summary_json: Path
    figure_png: Path
    figure_svg: Path


@dataclass(frozen=True)
class Mesh:
    points: list[tuple[float, float]]
    times: list[int]
    triangles: list[tuple[int, int, int, str]]
    spacelike_edges: list[tuple[int, int]]
    timelike_edges: list[tuple[int, int]]


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "Cargo.toml").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
    message = "Run this notebook from inside the causal-triangulations repository."
    raise RuntimeError(message)


def run_command(command: list[str], *, cwd: Path, timeout: int = 180) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(  # noqa: S603 - this notebook intentionally wraps the repository binary with fixed argv lists.
        command,
        cwd=cwd,
        text=True,
        capture_output=True,
        timeout=timeout,
        check=False,
    )
    if result.returncode != 0:
        command_text = " ".join(command)
        raise RuntimeError(f"command failed with exit code {result.returncode}: {command_text}\nstdout:\n{result.stdout}\nstderr:\n{result.stderr}")
    return result


def cdt_binary_path(root: Path) -> Path:
    configured = os.environ.get("CDT_BINARY")
    if configured is not None:
        path = Path(configured).expanduser().resolve()
        if path.is_file():
            return path
        message = f"CDT_BINARY does not point to a file: {path}"
        raise FileNotFoundError(message)

    binary_candidates = [root / "target" / "release" / name for name in ("cdt", "cdt.exe")]
    run_command(["cargo", "build", "--locked", "--release", "--bin", "cdt"], cwd=root, timeout=600)
    binary = next((candidate for candidate in binary_candidates if candidate.is_file()), binary_candidates[0])
    if not binary.is_file():
        expected_paths = ", ".join(str(candidate) for candidate in binary_candidates)
        message = f"cdt binary was not produced at any expected path: {expected_paths}"
        raise FileNotFoundError(message)
    return binary


def read_json(path: Path) -> dict[str, Any]:
    if not path.is_file():
        message = f"summary JSON not found: {path}"
        raise FileNotFoundError(message)
    loaded = json.loads(path.read_text(encoding=UTF8))
    if not isinstance(loaded, dict):
        message = f"summary JSON should be an object: {path}"
        raise TypeError(message)
    return loaded


ROOT = find_repo_root(Path.cwd().resolve())
RUN_PATHS = RunPaths(
    output_dir=ROOT / "target" / "notebooks" / "visualization",
    trace_csv=ROOT / "target" / "notebooks" / "visualization" / "trace.csv",
    summary_json=ROOT / "target" / "notebooks" / "visualization" / "summary.json",
    figure_png=ROOT / "target" / "notebooks" / "visualization" / "cdt_spacetime.png",
    figure_svg=ROOT / "target" / "notebooks" / "visualization" / "cdt_spacetime.svg",
)
RUN_PATHS.output_dir.mkdir(parents=True, exist_ok=True)
CDT_BINARY = cdt_binary_path(ROOT)
print(f"Using cdt binary: {CDT_BINARY}")

## 2. Run a small deterministic simulation

The run is intentionally modest so the notebook works as an example on laptops, CI, and Open OnDemand-backed Jupyter sessions. The cosmological constant is tuned for a short unfixed-volume run, not a production ensemble.

In [ ]:
SIMULATION_PARAMETERS = {
    "dimension": 2,
    "vertices_per_slice": 24,
    "timeslices": 7,
    "topology": "open-boundary",
    "cosmological_constant": 0.9,
    "steps": 160,
    "thermalization_steps": 20,
    "measurement_frequency": 20,
    "seed": 20260612,
}

command = [
    str(CDT_BINARY),
    "--dimension",
    str(SIMULATION_PARAMETERS["dimension"]),
    "--vertices-per-slice",
    str(SIMULATION_PARAMETERS["vertices_per_slice"]),
    "--timeslices",
    str(SIMULATION_PARAMETERS["timeslices"]),
    "--topology",
    str(SIMULATION_PARAMETERS["topology"]),
    "--cosmological-constant",
    str(SIMULATION_PARAMETERS["cosmological_constant"]),
    "--steps",
    str(SIMULATION_PARAMETERS["steps"]),
    "--thermalization-steps",
    str(SIMULATION_PARAMETERS["thermalization_steps"]),
    "--measurement-frequency",
    str(SIMULATION_PARAMETERS["measurement_frequency"]),
    "--seed",
    str(SIMULATION_PARAMETERS["seed"]),
    "--simulate",
    "--output-csv",
    str(RUN_PATHS.trace_csv),
    "--output-json",
    str(RUN_PATHS.summary_json),
]

result = run_command(command, cwd=ROOT, timeout=180)
print(result.stdout.strip())



## 3. Load trace and exported mesh data

The trace is a rectangular CSV suitable for Polars. Connectivity comes only from the summary's exported final mesh: a slab-triangle profile records counts, not vertex identities or simplex incidence, and therefore cannot reconstruct a spacetime mesh.

In [ ]:
def read_trace(path: Path) -> pl.DataFrame:
    if not path.is_file():
        message = f"trace CSV not found: {path}"
        raise FileNotFoundError(message)
    trace = pl.read_csv(path).with_columns(
        pl.col("accepted").cast(pl.Boolean),
        pl.col("proposed").cast(pl.Boolean),
    )
    required_columns = {"step", "accepted", "action", "vertices", "triangles"}
    missing = required_columns.difference(trace.columns)
    if missing:
        message = f"trace CSV is missing required columns: {sorted(missing)}"
        raise ValueError(message)
    if trace.height == 0:
        message = f"trace CSV is empty: {path}"
        raise ValueError(message)
    return trace


trace = read_trace(RUN_PATHS.trace_csv)
summary = read_json(RUN_PATHS.summary_json)

trace.select(
    pl.len().alias("rows"),
    pl.col("accepted").mean().alias("acceptance_rate"),
    pl.col("action").mean().alias("mean_action"),
    pl.col("vertices").last().alias("final_vertices"),
    pl.col("triangles").last().alias("final_triangles"),
)



## 4. Render the final CDT spacetime strip

Each horizontal row is a spatial slice and adjacent rows are connected by up/down CDT triangles from the exported final mesh. The vertical plotting coordinate is the exported foliation label, not the raw Delaunay embedding coordinate, so the picture displays causal time directly. The action and volume panels are intentionally diagnostic; suspiciously flat traces are evidence to investigate with `02_analysis_caches.ipynb`, not something to hide.

In [ ]:
def add_edge(edges: set[tuple[int, int]], first: int, second: int) -> None:
    if first != second:
        edge = (first, second) if first < second else (second, first)
        edges.add(edge)



In [ ]:
def triangle_kind(times: list[int], triangle: tuple[int, int, int]) -> str:
    labels = [times[index] for index in triangle]
    lower = min(labels)
    upper = max(labels)
    if upper - lower != 1:
        message = f"CDT simplex should span exactly two adjacent slices: {labels}"
        raise ValueError(message)
    return "up" if labels.count(lower) == 2 else "down"


DELAUNAY_MESH_SCHEMA = "delaunay.simplicial_complex"
DELAUNAY_MESH_SCHEMA_VERSION = 1
DELAUNAY_MESH_DIMENSION = 2
SUPPORTED_CDT_TOPOLOGY = "open-boundary"
U32_MAX = (1 << 32) - 1
SAMPLE_VERTEX_IDS = [
    "00000000-0000-0000-0000-000000000001",
    "00000000-0000-0000-0000-000000000002",
    "00000000-0000-0000-0000-000000000003",
]
SAMPLE_SIMPLEX_ID = "00000000-0000-0000-0000-000000000004"


def parse_nonnegative_integer(raw_value: Any, *, field: str) -> int:
    if isinstance(raw_value, bool) or not isinstance(raw_value, int):
        message = f"{field} should be a non-negative integer: {raw_value}"
        raise TypeError(message)
    if raw_value < 0:
        message = f"{field} should be non-negative: {raw_value}"
        raise ValueError(message)
    return raw_value


def parse_finite_number(raw_value: Any, *, field: str) -> float:
    if isinstance(raw_value, bool) or not isinstance(raw_value, (int, float)):
        message = f"{field} should be a finite JSON number: {raw_value}"
        raise TypeError(message)
    parsed = float(raw_value)
    if not math.isfinite(parsed):
        message = f"{field} should be finite: {raw_value}"
        raise ValueError(message)
    return parsed


def parse_stable_uuid(raw_id: Any, *, field: str) -> str:
    if not isinstance(raw_id, str):
        message = f"{field} should be a canonical UUID string: {raw_id}"
        raise TypeError(message)
    try:
        parsed = UUID(raw_id)
    except ValueError as error:
        message = f"{field} should be a canonical UUID string: {raw_id}"
        raise ValueError(message) from error
    canonical = str(parsed)
    if parsed.int == 0 or raw_id != canonical:
        message = f"{field} should be a canonical non-nil UUID string: {raw_id}"
        raise ValueError(message)
    return canonical


def parse_mesh_metadata(metadata: dict[str, Any]) -> tuple[int, int]:
    schema = metadata.get("schema")
    if schema != DELAUNAY_MESH_SCHEMA:
        message = f"unsupported final mesh schema: {schema}"
        raise ValueError(message)
    schema_version = parse_nonnegative_integer(metadata.get("schema_version"), field="mesh schema version")
    if schema_version != DELAUNAY_MESH_SCHEMA_VERSION:
        message = f"unsupported {DELAUNAY_MESH_SCHEMA} version: {schema_version}"
        raise ValueError(message)
    dimension = parse_nonnegative_integer(metadata.get("dimension"), field="mesh dimension")
    if dimension != DELAUNAY_MESH_DIMENSION:
        message = f"unsupported {DELAUNAY_MESH_SCHEMA} dimension: {dimension}"
        raise ValueError(message)
    vertex_count = parse_nonnegative_integer(metadata.get("vertex_count"), field="mesh vertex count")
    simplex_count = parse_nonnegative_integer(metadata.get("simplex_count"), field="mesh simplex count")
    return vertex_count, simplex_count


def exported_mesh_payload(summary: dict[str, Any]) -> tuple[list[Any], list[Any], list[Any], int] | None:
    final = summary.get("final_triangulation")
    if not isinstance(final, dict):
        return None
    exported_mesh = final.get("mesh")
    if not isinstance(exported_mesh, dict):
        return None
    metadata = exported_mesh.get("metadata")
    if not isinstance(metadata, dict):
        return None
    topology = final.get("topology")
    if topology != SUPPORTED_CDT_TOPOLOGY:
        message = f"this spacetime-strip notebook does not support topology: {topology}"
        raise ValueError(message)
    time_slices = parse_nonnegative_integer(final.get("time_slices"), field="CDT time-slice count")
    if not 2 <= time_slices <= U32_MAX:
        message = f"CDT spacetime strip needs between 2 and {U32_MAX} time slices: {time_slices}"
        raise ValueError(message)
    vertex_count, simplex_count = parse_mesh_metadata(metadata)
    exported_vertices = exported_mesh.get("vertices")
    exported_simplices = exported_mesh.get("simplices")
    exported_adjacency = exported_mesh.get("adjacency")
    vertex_time_labels = final.get("vertex_time_labels")
    if not isinstance(exported_vertices, list) or not isinstance(exported_simplices, list):
        message = "Delaunay mesh vertices and simplices should be arrays"
        raise TypeError(message)
    if not isinstance(exported_adjacency, list):
        message = "Delaunay mesh adjacency should be an array"
        raise TypeError(message)
    if not isinstance(vertex_time_labels, list):
        message = "CDT vertex time labels should be an array"
        raise TypeError(message)
    if vertex_count != len(exported_vertices) or simplex_count != len(exported_simplices):
        message = (
            "Delaunay mesh metadata counts do not match entity arrays: "
            f"vertices {vertex_count} != {len(exported_vertices)}, "
            f"simplices {simplex_count} != {len(exported_simplices)}"
        )
        raise ValueError(message)
    if not exported_vertices or not exported_simplices:
        message = "CDT visualization requires a non-empty vertex and simplex export"
        raise ValueError(message)
    return exported_vertices, exported_simplices, vertex_time_labels, time_slices


def parse_vertex_time(vertex_id: str, raw_time: Any) -> int:
    if isinstance(raw_time, bool):
        message = f"exported mesh vertex {vertex_id} has non-numeric time label: {raw_time}"
        raise TypeError(message)
    if isinstance(raw_time, int):
        parsed_time = raw_time
    elif isinstance(raw_time, float) and raw_time.is_integer():
        parsed_time = int(raw_time)
    else:
        message = f"exported mesh vertex {vertex_id} has non-numeric time label: {raw_time}"
        raise TypeError(message)
    if not 0 <= parsed_time <= U32_MAX:
        message = f"exported mesh vertex {vertex_id} has time label outside the u32 range: {parsed_time}"
        raise ValueError(message)
    return parsed_time


if parse_vertex_time("vertex-0", 2.0) != 2:
    message = "expected numeric time labels to parse as integers"
    raise AssertionError(message)
for bad_time in ("2", True, 2.5, None):
    try:
        parse_vertex_time("vertex-2", bad_time)
    except TypeError as error:
        nonnumeric_error = str(error)
    else:
        nonnumeric_error = ""
    if "non-numeric time label" not in nonnumeric_error:
        message = "expected non-numeric time labels to fail with a clear diagnostic"
        raise AssertionError(message)
for out_of_range_time in (-1, U32_MAX + 1):
    try:
        parse_vertex_time(SAMPLE_VERTEX_IDS[0], out_of_range_time)
    except ValueError as error:
        range_error = str(error)
    else:
        range_error = ""
    if "outside the u32 range" not in range_error:
        message = "expected out-of-range time labels to fail with a clear diagnostic"
        raise AssertionError(message)
for invalid_uuid in ("vertex-0", "00000000-0000-0000-0000-000000000000", None):
    try:
        parse_stable_uuid(invalid_uuid, field="test vertex id")
    except (TypeError, ValueError):  # fmt: skip
        pass
    else:
        message = f"expected invalid UUID to be rejected: {invalid_uuid}"
        raise AssertionError(message)


def parse_vertex_time_labels(vertex_time_labels: list[Any], *, time_slices: int) -> dict[str, int]:
    time_by_vertex_id: dict[str, int] = {}
    for label in vertex_time_labels:
        if not isinstance(label, dict):
            message = "CDT vertex time label should be an object"
            raise TypeError(message)
        vertex_id = parse_stable_uuid(label.get("vertex_id"), field="CDT time-label vertex id")
        if vertex_id in time_by_vertex_id:
            message = f"duplicate CDT time label for vertex {vertex_id}"
            raise ValueError(message)
        time_label = parse_vertex_time(vertex_id, label.get("time"))
        if time_label >= time_slices:
            message = f"vertex {vertex_id} time label {time_label} exceeds {time_slices} slices"
            raise ValueError(message)
        time_by_vertex_id[vertex_id] = time_label
    return time_by_vertex_id


try:
    parse_vertex_time_labels(
        [{"vertex_id": SAMPLE_VERTEX_IDS[0], "time": 2}],
        time_slices=2,
    )
except ValueError as error:
    slice_range_error = str(error)
else:
    slice_range_error = ""
if "exceeds 2 slices" not in slice_range_error:
    message = "expected labels outside the declared slice count to be rejected"
    raise AssertionError(message)


def parse_exported_vertices(exported_vertices: list[Any], time_by_vertex_id: dict[str, int]) -> tuple[list[tuple[float, float]], list[int], dict[str, int]]:
    points: list[tuple[float, float]] = []
    times: list[int] = []
    vertex_indices: dict[str, int] = {}
    for vertex in exported_vertices:
        if not isinstance(vertex, dict):
            message = "exported mesh vertex should be an object"
            raise TypeError(message)
        vertex_id = parse_stable_uuid(vertex.get("id"), field="exported mesh vertex id")
        if vertex_id in vertex_indices:
            message = f"duplicate exported mesh vertex UUID: {vertex_id}"
            raise ValueError(message)
        coordinates = vertex.get("coordinates")
        if not isinstance(coordinates, list) or len(coordinates) != 2:
            message = f"exported mesh vertex {vertex_id} needs two coordinates"
            raise ValueError(message)
        spatial_coordinate = parse_finite_number(coordinates[0], field=f"vertex {vertex_id} spatial coordinate")
        parse_finite_number(coordinates[1], field=f"vertex {vertex_id} temporal coordinate")
        if vertex_id not in time_by_vertex_id:
            message = f"exported mesh vertex {vertex_id} has no CDT time label"
            raise ValueError(message)
        time_label = time_by_vertex_id[vertex_id]
        vertex_indices[vertex_id] = len(points)
        points.append((spatial_coordinate, float(time_label)))
        times.append(time_label)

    unknown_time_ids = sorted(set(time_by_vertex_id) - set(vertex_indices))
    if unknown_time_ids:
        message = f"CDT time labels reference unknown mesh vertices: {unknown_time_ids}"
        raise ValueError(message)
    return points, times, vertex_indices


def parse_exported_simplices(
    exported_simplices: list[Any], vertex_indices: dict[str, int], times: list[int]
) -> tuple[list[tuple[int, int, int, str]], set[tuple[int, int]], set[tuple[int, int]]]:
    triangles: list[tuple[int, int, int, str]] = []
    spacelike: set[tuple[int, int]] = set()
    timelike: set[tuple[int, int]] = set()
    simplex_ids: set[str] = set()
    for simplex in exported_simplices:
        if not isinstance(simplex, dict):
            message = "exported mesh simplex should be an object"
            raise TypeError(message)
        simplex_id = parse_stable_uuid(simplex.get("id"), field="exported mesh simplex id")
        if simplex_id in simplex_ids:
            message = f"duplicate exported mesh simplex UUID: {simplex_id}"
            raise ValueError(message)
        simplex_ids.add(simplex_id)
        raw_vertex_ids = simplex.get("vertex_ids")
        if not isinstance(raw_vertex_ids, list) or len(raw_vertex_ids) != 3:
            message = f"exported mesh simplex should have three vertex UUIDs: {simplex}"
            raise ValueError(message)
        vertex_ids = [parse_stable_uuid(vertex_id, field=f"simplex {simplex_id} vertex id") for vertex_id in raw_vertex_ids]
        if len(set(vertex_ids)) != 3:
            message = f"exported mesh simplex should reference three distinct vertices: {simplex}"
            raise ValueError(message)
        missing_vertex_ids = [vertex_id for vertex_id in vertex_ids if vertex_id not in vertex_indices]
        if missing_vertex_ids:
            message = f"exported mesh simplex references unknown vertices: {missing_vertex_ids}"
            raise ValueError(message)
        triangle = (
            vertex_indices[vertex_ids[0]],
            vertex_indices[vertex_ids[1]],
            vertex_indices[vertex_ids[2]],
        )
        triangles.append((*triangle, triangle_kind(times, triangle)))
        for first, second in ((triangle[0], triangle[1]), (triangle[1], triangle[2]), (triangle[2], triangle[0])):
            target = spacelike if times[first] == times[second] else timelike
            add_edge(target, first, second)
    return triangles, spacelike, timelike



In [ ]:
try:
    parse_exported_vertices(
        [{"id": SAMPLE_VERTEX_IDS[0], "coordinates": [0.0, float("nan")]}],
        {SAMPLE_VERTEX_IDS[0]: 0},
    )
except ValueError as error:
    nonfinite_coordinate_error = str(error)
else:
    nonfinite_coordinate_error = ""
if "should be finite" not in nonfinite_coordinate_error:
    message = "expected every exported coordinate component to be finite"
    raise AssertionError(message)
sample_vertex_indices = {vertex_id: index for index, vertex_id in enumerate(SAMPLE_VERTEX_IDS)}
for invalid_vertex_ids, invalid_times, expected_text in (
    ([SAMPLE_VERTEX_IDS[0], SAMPLE_VERTEX_IDS[0], SAMPLE_VERTEX_IDS[2]], [0, 0, 1], "three distinct vertices"),
    (SAMPLE_VERTEX_IDS, [0, 0, 0], "two adjacent slices"),
    (SAMPLE_VERTEX_IDS, [0, 1, 2], "two adjacent slices"),
):
    try:
        parse_exported_simplices(
            [{"id": SAMPLE_SIMPLEX_ID, "vertex_ids": invalid_vertex_ids}],
            sample_vertex_indices,
            invalid_times,
        )
    except ValueError as error:
        simplex_error = str(error)
    else:
        simplex_error = ""
    if expected_text not in simplex_error:
        message = f"expected malformed simplex to be rejected: {invalid_vertex_ids}, {invalid_times}"
        raise AssertionError(message)



In [ ]:
def mesh_from_summary(summary: dict[str, Any]) -> Mesh | None:
    payload = exported_mesh_payload(summary)
    if payload is None:
        return None
    exported_vertices, exported_simplices, vertex_time_labels, time_slices = payload
    time_by_vertex_id = parse_vertex_time_labels(vertex_time_labels, time_slices=time_slices)
    points, times, vertex_indices = parse_exported_vertices(exported_vertices, time_by_vertex_id)
    triangles, spacelike, timelike = parse_exported_simplices(exported_simplices, vertex_indices, times)

    return Mesh(
        points=points,
        times=times,
        triangles=triangles,
        spacelike_edges=sorted(spacelike),
        timelike_edges=sorted(timelike),
    )



In [ ]:
def scaled_for_display(mesh: Mesh) -> Mesh:
    x_values = [point[0] for point in mesh.points]
    y_values = [point[1] for point in mesh.points]
    x_span = max(x_values) - min(x_values)
    y_span = max(y_values) - min(y_values)
    if x_span <= 0.0 or y_span <= 0.0:
        return mesh
    midpoint = (max(x_values) + min(x_values)) / 2.0
    target_x_span = 1.75 * y_span
    scale = max(1.0, min(18.0, target_x_span / x_span))
    scaled_points = [((x - midpoint) * scale, y) for x, y in mesh.points]
    return Mesh(
        points=scaled_points,
        times=mesh.times,
        triangles=mesh.triangles,
        spacelike_edges=mesh.spacelike_edges,
        timelike_edges=mesh.timelike_edges,
    )


def edge_segments(mesh: Mesh, edges: list[tuple[int, int]]) -> list[tuple[tuple[float, float], tuple[float, float]]]:
    return [(mesh.points[first], mesh.points[second]) for first, second in edges]


def triangle_polygons(mesh: Mesh, kind: str) -> list[list[tuple[float, float]]]:
    polygons: list[list[tuple[float, float]]] = []
    for first, second, third, triangle_kind in mesh.triangles:
        if triangle_kind == kind:
            polygons.append([mesh.points[first], mesh.points[second], mesh.points[third]])
    return polygons


def scalar_from_trace(trace: pl.DataFrame, column: str) -> list[float]:
    return [float(value) for value in trace.get_column(column).to_list()]


def render_spacetime(mesh: Mesh, trace: pl.DataFrame, summary: dict[str, Any], paths: RunPaths) -> tuple[Path, Path]:
    aggregate = summary.get("aggregate", {})
    final = summary.get("final_triangulation", {})
    acceptance = float(trace.select(pl.col("accepted").mean()).item())
    final_vertices = int(final.get("vertices", trace.get_column("vertices")[-1]))
    final_triangles = int(final.get("triangles", trace.get_column("triangles")[-1]))
    average_action = aggregate.get("average_action", trace.select(pl.col("action").mean()).item())

    fig = plt.figure(figsize=(14, 8), dpi=200, facecolor="#070912")
    grid = fig.add_gridspec(2, 5, width_ratios=[1.3, 1.3, 1.3, 1.3, 1.0], wspace=0.32, hspace=0.34)
    mesh_axis = fig.add_subplot(grid[:, :4])
    action_axis = fig.add_subplot(grid[0, 4])
    volume_axis = fig.add_subplot(grid[1, 4])

    for axis in (mesh_axis, action_axis, volume_axis):
        axis.set_facecolor("#070912")
        axis.tick_params(colors="#c8d3ea", labelsize=8)
        for spine in axis.spines.values():
            spine.set_color("#2a3558")

    mesh_axis.add_collection(PolyCollection(triangle_polygons(mesh, "up"), facecolors="#39c5d9", edgecolors="#77f7ff", linewidths=0.35, alpha=0.24))
    mesh_axis.add_collection(PolyCollection(triangle_polygons(mesh, "down"), facecolors="#ff7b45", edgecolors="#ffc06d", linewidths=0.35, alpha=0.24))
    mesh_axis.add_collection(LineCollection(edge_segments(mesh, mesh.timelike_edges), colors="#89a8ff", linewidths=0.55, alpha=0.54))
    mesh_axis.add_collection(LineCollection(edge_segments(mesh, mesh.spacelike_edges), colors="#ffe28a", linewidths=0.72, alpha=0.72))

    x_values = [point[0] for point in mesh.points]
    y_values = [point[1] for point in mesh.points]
    mesh_axis.scatter(x_values, y_values, c=mesh.times, cmap="viridis", s=18, edgecolors="#f5f7ff", linewidths=0.25, zorder=4)
    mesh_axis.set_aspect("equal", adjustable="datalim")
    mesh_axis.margins(x=0.08, y=0.06)
    mesh_axis.set_axis_off()
    mesh_axis.set_title("1+1 CDT spacetime triangulation", color="#f8fbff", fontsize=18, pad=18)
    config = summary.get("config", {})
    topology = config.get("topology", SIMULATION_PARAMETERS["topology"]) if isinstance(config, dict) else SIMULATION_PARAMETERS["topology"]
    mesh_axis.text(
        0.01,
        0.02,
        f"{topology} run | seed {SIMULATION_PARAMETERS['seed']} | acceptance {acceptance:.2%} | final V={final_vertices}, T={final_triangles}",
        transform=mesh_axis.transAxes,
        color="#dbe8ff",
        fontsize=9,
        alpha=0.9,
    )

    steps = scalar_from_trace(trace, "step")
    action_axis.plot(steps, scalar_from_trace(trace, "action"), color="#77f7ff", linewidth=1.7)
    action_axis.set_title("Action", color="#f8fbff", fontsize=11)
    action_axis.set_xlabel("step", color="#c8d3ea", fontsize=8)
    action_axis.grid(color="#233052", alpha=0.5, linewidth=0.6)

    volume_axis.plot(steps, scalar_from_trace(trace, "vertices"), color="#ffc06d", linewidth=1.7, label="vertices")
    volume_axis.plot(steps, scalar_from_trace(trace, "triangles"), color="#c79bff", linewidth=1.2, label="triangles")
    volume_axis.set_title("Volume", color="#f8fbff", fontsize=11)
    volume_axis.set_xlabel("step", color="#c8d3ea", fontsize=8)
    volume_axis.legend(loc="best", fontsize=7, frameon=False, labelcolor="#dbe8ff")
    volume_axis.grid(color="#233052", alpha=0.5, linewidth=0.6)

    fig.suptitle(f"causal-triangulations example visualization | mean action {float(average_action):.3f}", color="#b8c7ff", fontsize=10, y=0.98)
    paths.figure_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(paths.figure_png, bbox_inches="tight", facecolor=fig.get_facecolor())
    fig.savefig(paths.figure_svg, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    return paths.figure_png, paths.figure_svg


mesh = mesh_from_summary(summary)
if mesh is None:
    message = "summary does not contain the supported Delaunay mesh schema; "
    message += "profile counts cannot reconstruct mesh connectivity; rerun with the current checkout"
    raise ValueError(message)
render_spacetime(scaled_for_display(mesh), trace, summary, RUN_PATHS)

## 5. Use the generated image

The rendered files are written to `target/notebooks/visualization/cdt_spacetime.png` and `target/notebooks/visualization/cdt_spacetime.svg`. Try changing `SIMULATION_PARAMETERS["seed"]`, `SIMULATION_PARAMETERS["cosmological_constant"]`, or the number of time slices to explore different short-run geometries.